# Open a QAChat RAG project

This tutorial assembles the earlier layers into one user-facing RAG
session. A project keeps its source location, settings, reusable indexes,
and conversation log together.

## 1. Prepare QAChat

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Choose a project

Use **code** or **literature** to open a committed toy project.

To use your own material, copy the matching sandbox project, replace its
source files or change **source** in **profile.ini**, and set
**PROJECT_DIRECTORY** to that project folder.

In [ ]:
PROJECT_KIND = "code"
PROJECT_DIRECTORY = None

# PROJECT_DIRECTORY = "/path/to/my-qachat-project"

In [ ]:
if PROJECT_DIRECTORY is None:
    project_directory = root / "sandbox" / PROJECT_KIND
else:
    project_directory = Path(PROJECT_DIRECTORY).expanduser()

print(f"Project: {project_directory}")

## 3. Inspect the project

**profile.ini** selects the source, models, chunk sizes, retrieval count,
temperature, and other reusable settings. Generated files remain inside
the selected project and are excluded from Git.

In [ ]:
from qachat import open_project

project = open_project(project_directory)

print(f"Kind: {project.profile.kind}")
print(f"Source: {project.source}")
print(f"Indexes: {project.indexes}")
print(f"Logs: {project.logs}")

In [ ]:
profile_path = project.root / "profile.ini"
print(profile_path.read_text(encoding="utf-8"))

## 4. Assemble the RAG application

**RAGSession** loads the models, reads the source, and prepares all
retrieval channels. The first open builds the indexes. Later opens reuse
them when the source and index settings have not changed.

In [ ]:
from qachat import RAGSession

if "session" in globals():
    session.close()

session = RAGSession(project_directory)

In [ ]:
print(f"Backend: {session.backend}")
print(f"Index status: {session.index_status}")
print(f"Restored turns: {len(session.turns)}")

## 5. Ask a grounded question

QAChat searches semantic, lexical, and code-aware indexes, combines their
rankings, expands explicit repository references, and gives the selected
evidence to the chat model.

In [ ]:
DEFAULT_QUESTIONS = {
    "code": "How does this project convert Celsius to Kelvin?",
    "literature": "What did the toy sleep study observe?",
}

QUESTION = DEFAULT_QUESTIONS[project.profile.kind]
print(QUESTION)

In [ ]:
answer = session.ask(QUESTION)

print(answer)

## 6. Inspect the evidence

The saved turn contains the exact chunks supplied to the model. Citation
numbers here match bracketed citations requested in the answer.

In [ ]:
turn = session.turns[-1]

for item in turn.evidence:
    location = f"{item.path}:{item.start_line}-{item.end_line}"
    print(f"[{item.citation}] {location}")
    print(item.text)
    print()

## 7. Inspect the saved state

The index can be rebuilt from source. The conversation log records every
question, response, evidence set, and exact parameter snapshot.

In [ ]:
print(project.indexes / "retrieval.json")
print(project.logs / "conversation.json")

In [ ]:
import json

print(json.dumps(turn.parameters, indent=2, sort_keys=True))

## 8. Reopen the project

Close and reopen the same project to verify that its index and conversation
state are restored. The models are loaded again, while current indexes are
read from disk.

In [ ]:
session.close()
session = RAGSession(project_directory)

print(f"Index status: {session.index_status}")
print(f"Restored turns: {len(session.turns)}")

In [ ]:
for number, saved_turn in enumerate(session.turns, start=1):
    print(f"Question {number}: {saved_turn.question}")
    print(f"Answer {number}: {saved_turn.answer}")

## 9. Start a new conversation when needed

Resetting renames the active log with a UTC timestamp before clearing the
conversation from memory. Set this option to **True** when you want a new
conversation. Reusable indexes are not changed.

In [ ]:
START_NEW_CONVERSATION = False

if START_NEW_CONVERSATION:
    archived_path = session.reset()
    print(f"Archived conversation: {archived_path}")
    print(f"Active turns: {len(session.turns)}")

## 10. Release the models

In [ ]:
session.close()

print("RAG session closed.")

## Result

One configured folder now behaves as a reusable QAChat project. Opening it
restores its retrieval indexes, full conversation, evidence, and settings.